In [ ]:
# @title Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Using the Vertex AI MCP Server

<table align="left">
  <td style="text-align: center">
    <a href="https://colab.research.google.com/github/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/official/mcp/vertex_ai_mcp_server.ipynb">
      <img width="32px" src="https://www.gstatic.com/pantheon/images/bigquery/welcome_page/colab-logo.svg" alt="Google Colaboratory logo"><br> Open in Colab
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/colab/import/https:%2F%2Fraw.githubusercontent.com%2FGoogleCloudPlatform%2Fvertex-ai-samples%2Fmain%2Fnotebooks%2Fofficial%2Fmcp%2Fvertex_ai_mcp_server.ipynb">
      <img width="32px" src="https://lh3.googleusercontent.com/JmcxdQi-qOpctIvWKgPtrzZdJJK-J3sWE1RsfjZNwshCFgE_9fULcNpuXYTilIR2hjwN" alt="Google Cloud Colab Enterprise logo"><br> Open in Colab Enterprise
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://console.cloud.google.com/vertex-ai/workbench/deploy-notebook?download_url=https://raw.githubusercontent.com/GoogleCloudPlatform/vertex-ai-samples/main/notebooks/official/mcp/vertex_ai_mcp_server.ipynb">
      <img src="https://www.gstatic.com/images/branding/gcpiconscolors/vertexai/v1/32px.svg" alt="Vertex AI logo"><br> Open in Vertex AI Workbench
    </a>
  </td>
  <td style="text-align: center">
    <a href="https://github.com/GoogleCloudPlatform/vertex-ai-samples/blob/main/notebooks/official/mcp/vertex_ai_mcp_server.ipynb">
      <img width="32px" src="https://raw.githubusercontent.com/primer/octicons/refs/heads/main/icons/mark-github-24.svg" alt="GitHub logo"><br> View on GitHub
    </a>
  </td>
</table>

<div style="clear: both;"></div>


| Authors |
| --- |
| [Eric Dong](www.linkedin.com/in/edong186) |

## Overview

This notebook demonstrates how to connect to the **Remote Model Context Protocol (MCP) server for Vertex AI** (https://us-central1-aiplatform.googleapis.com/mcp).

Unlike local MCP servers that run as subprocesses, this server is hosted by Google Cloud. It allows LLMs to interact directly with your Google Cloud infrastructure, specifically Endpoint Management, Prompt Management, and Model Garden, using natural language.

## Getting Started

### Install libraries

In [1]:
%pip install --upgrade --quiet google-auth google-genai mcp jq

> ⚠️ Note: You can ignore the pip's dependency errors.

### Import libraries


In [2]:
import os
import sys

import google.auth
from google import genai
from google.auth.transport.requests import Request
from google.genai import types
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

### Authenticate your notebook environment

If you are running this notebook on Google Colab, run the cell below to authenticate your environment.

In [4]:
if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()

### Autenticate your Google Cloud Project for Vertex AI

You can use a Google Cloud Project or an API Key for authentication. This tutorial uses a Google Cloud Project.

- [Enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com)

In [5]:
# fmt: off
# TODO: replace with template
PROJECT_ID = "[your-project-id]"  # @param {type: "string", placeholder: "[your-project-id]", isTemplate: true}
# fmt: on
if not PROJECT_ID or PROJECT_ID == "[your-project-id]":
    PROJECT_ID = str(os.environ.get("GOOGLE_CLOUD_PROJECT"))

LOCATION = "us-central1"

client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

## Use the Vertex AI MCP endpoint

In [6]:
MCP_URL = "https://us-central1-aiplatform.googleapis.com/mcp"  # @param {type: "string"}

## Enable MCP for your Project

⚠️ You will encounter a 403 Forbidden or Permission Denied error if the MCP feature is enabled for your specific service.


In [7]:
os.environ["CLOUDSDK_CORE_PROJECT"] = PROJECT_ID

!gcloud beta services mcp enable aiplatform.googleapis.com

## Get the access token to authenticate against the MCP endpoint

- Define the explicit scope required for Google Cloud APIs
- Pass the scopes into the `auth.default()` method
- The token will expire in 60 minutes by default.

In [8]:
def get_auth_token():
    scopes = ["https://www.googleapis.com/auth/cloud-platform"]
    credentials, _ = google.auth.default(scopes=scopes)
    credentials.refresh(Request())
    return credentials.token


token = get_auth_token()

## Using the MCP Server

### **Example 1**: Use cURL

This method is for low-level discovery, particularly useful for debugging.

#### Push the variables we generated earlier into the system environment

In [9]:
os.environ["TOKEN"] = token
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["MCP_URL"] = MCP_URL

#### Send an exact JSON-RPC payload to initialize the MCP connection

- `X-Goog-User-Project` header is mandatory for billing and quota routing.

In [10]:
%%bash

curl -N -X POST "$MCP_URL" \
  -H "Authorization: Bearer $TOKEN" \
  -H "X-Goog-User-Project: $PROJECT_ID" \
  -H "Content-Type: application/json" \
  -H 'accept: application/json, text/event-stream' \
  -d '{
    "method": "tools/list",
    "jsonrpc": "2.0",
    "id": 1
  }'  2>/dev/null >response.json

jq -r ".result.tools[].name" response.json

create_endpoint
get_endpoint
list_endpoints
update_endpoint
delete_endpoint
create_prompt
get_prompt
list_prompts
update_prompt
delete_prompt
get_publisher_model
list_publisher_models
search_publisher_models


### **Example 2**: Use SDK

This method is for agentic approach. The Gen AI SDK, Agent Developement Kit(ADK) and other agent frameworks handle Automatic Function Calling (AFC) when you pass the session as a tool. The model will decide which tool to call based on your prompt.


In [11]:
# Initialize a Gen AI client
client = genai.Client(vertexai=True, project=PROJECT_ID, location="global")


async def run_mcp_agent(user_prompt: str):
    headers = {
        "Authorization": f"Bearer {get_auth_token()}",
        "X-Goog-User-Project": PROJECT_ID,
    }

    # streamablehttp_client manages the persistent connection to the remote MCP endpoint
    async with streamablehttp_client(url=MCP_URL, headers=headers) as streams:
        async with ClientSession(streams[0], streams[1]) as session:
            await session.initialize()

            tools = await session.list_tools()
            print("Available MCP Tools:")
            for tool in tools.tools:
                print(f"- {tool.name}: {tool.description}")

            print(f"🤖 User Prompt: {user_prompt}\n")

            # The 'tools=[session]' allows Gemini to 'see' and execute MCP tools
            response = await client.aio.models.generate_content(
                model="gemini-3.1-flash-lite-preview",
                contents=user_prompt,
                config=types.GenerateContentConfig(
                    tools=[session],
                ),
            )

            print("--- Model Response ---")
            print(response.text)


# Execute the agent
prompt = f"List my Vertex AI endpoints in project {PROJECT_ID} and location {LOCATION} and check if any managed prompts exist."

await run_mcp_agent(prompt)

Available MCP Tools:
- create_endpoint: Creates an Endpoint.
- get_endpoint: Gets an Endpoint.
- list_endpoints: Lists Endpoints in a Location.
- update_endpoint: Updates an Endpoint.
- delete_endpoint: Deletes an Endpoint.
- create_prompt: Creates a new managed prompt. Saves the display_name, template text, system instructions, and inference settings (e.g. temperature). Use this to persist a prompt for future versioning.
- get_prompt: Retrieves full details of a prompt including metadata, labels, and configuration. Use this to inspect a prompt's settings or retrieve its full resource name before an update.
- list_prompts: Lists all managed prompts in a location. Use this to browse existing prompts or map a user-friendly display_name to a technical resource name.
- update_prompt: Updates an existing prompt's fields (display_name, instructions, settings, or labels). This performs a patch; fields not specified remain unchanged.
- delete_prompt: Deletes a Prompt.
- get_publisher_model: Re

## Learn More
You have successfully connected a Gemini model to the Vertex AI Remote MCP server to manage cloud resources using natural language. Explore the following resources:

- [Vertex AI MCP Reference](https://docs.cloud.google.com/vertex-ai/docs/reference/mcp)
- [Managing MCP Servers](https://docs.cloud.google.com/mcp/enable-disable-mcp-servers)